# Phase 2 - Manual Labeling Workbench (Jupyter 3)

Notebook para etiquetar segmentos manualmente de forma comoda y guardando en cada accion.

## Objetivo
- Navegar por ventanas temporales del grafico.
- Etiquetar segmentos de todo 2016 en forma manual.
- Ver candidatos automaticos en la ventana (opcional).
- Guardar incrementalmente sin perder progreso.



## Flujo sugerido (rapido)

1. Ejecuta todas las celdas de setup.
2. En la celda de UI, usa sliders/botones para navegar por 2016.
3. Para candidatos visibles: usa `local_id`, `label` y `Accept cand`.
4. Para segmentos manuales: define `start_ts` y `end_ts` y usa `Accept custom`.
5. Revisa progreso con `session_status()` y `list_labels()`.

Todo se guarda automaticamente en `output_manual_labels_2016`.



In [ ]:
import uuid
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except Exception:
    widgets = None
    HAS_WIDGETS = False

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 220)



In [ ]:
# =========================
# Config
# =========================
# Dataset 2016 completo (H1)
DATA_DIR = Path('./data_2016/parquet_h1')
SYMBOL = 'USDJPY'

# Catalogo de candidatos automaticos (opcional)
# Dejalo en None para trabajo 100% manual.
CANDIDATE_CATALOG_PATH = None
# CANDIDATE_CATALOG_PATH = Path('./phases/phase02_market_alphabet/output_segments_beyond_sota/segments_catalog.parquet')

# Trabajo manual (separado para 2016)
OUTPUT_DIR = Path('./phases/phase02_market_alphabet/output_manual_labels_2016')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ACCEPT_PATH_PARQUET = OUTPUT_DIR / f'{SYMBOL}_accepted.parquet'
ACCEPT_PATH_CSV = OUTPUT_DIR / f'{SYMBOL}_accepted.csv'
REJECT_PATH_PARQUET = OUTPUT_DIR / f'{SYMBOL}_rejected.parquet'
REJECT_PATH_CSV = OUTPUT_DIR / f'{SYMBOL}_rejected.csv'

# Navegacion
WINDOW_BARS = 240
WINDOW_STRIDE = 120
TOP_CANDIDATES_IN_VIEW = 35

# Estilo
PLOT_WIDTH = 17
PLOT_HEIGHT = 6



In [ ]:
# =========================
# Carga de datos
# =========================

def _asset_path(symbol: str) -> Path:
    p = DATA_DIR / f'{symbol.upper()}.parquet'
    if not p.exists():
        raise FileNotFoundError(f'No existe asset parquet: {p}')
    return p


def _load_asset(symbol: str) -> pd.DataFrame:
    p = _asset_path(symbol)
    df = pd.read_parquet(p).copy()
    need = ['timestamp', 'open', 'high', 'low', 'close']
    miss = [c for c in need if c not in df.columns]
    if miss:
        raise ValueError(f'Faltan columnas {miss} en {p}')

    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    for c in ['open', 'high', 'low', 'close']:
        df[c] = pd.to_numeric(df[c], errors='coerce')

    df = df.dropna(subset=['timestamp', 'close'])
    df = df.sort_values('timestamp').drop_duplicates('timestamp', keep='last').reset_index(drop=True)
    df['idx'] = np.arange(len(df), dtype=int)
    return df


def _load_candidates(symbol: str) -> pd.DataFrame:
    if CANDIDATE_CATALOG_PATH is None or str(CANDIDATE_CATALOG_PATH).strip() == '':
        print('[INFO] Catalogo de candidatos desactivado (manual puro).')
        return pd.DataFrame()

    p = Path(CANDIDATE_CATALOG_PATH)
    if not p.exists():
        print(f'[WARN] No existe catalogo de candidatos: {p}')
        return pd.DataFrame()

    c = pd.read_parquet(p)
    if 'symbol' not in c.columns:
        print('[WARN] Catalogo no tiene columna symbol.')
        return pd.DataFrame()

    c = c[c['symbol'].astype(str).str.upper() == symbol.upper()].copy()
    if c.empty:
        print(f'[WARN] No hay candidatos para {symbol} en catalogo.')
        return c

    for col in ['start_idx', 'end_idx']:
        c[col] = pd.to_numeric(c[col], errors='coerce').astype('Int64')
    c = c.dropna(subset=['start_idx', 'end_idx']).copy()
    c['start_idx'] = c['start_idx'].astype(int)
    c['end_idx'] = c['end_idx'].astype(int)

    if 'score' not in c.columns:
        c['score'] = 0.0
    if 'subtype' not in c.columns:
        c['subtype'] = 'candidate'
    if 'method' not in c.columns:
        c['method'] = 'unknown'

    c = c.sort_values('score', ascending=False).reset_index(drop=True)
    return c


def _load_or_init(path_parquet: Path, cols: list[str]) -> pd.DataFrame:
    if path_parquet.exists():
        d = pd.read_parquet(path_parquet)
        for c in cols:
            if c not in d.columns:
                d[c] = np.nan
        return d[cols].copy()
    return pd.DataFrame(columns=cols)


DF = _load_asset(SYMBOL)
CAND = _load_candidates(SYMBOL)

ACCEPT_COLS = [
    'row_id', 'symbol', 'source', 'candidate_row_id', 'method', 'subtype',
    'start_idx', 'end_idx', 'start_ts', 'end_ts', 'n_bars',
    'label', 'quality', 'notes', 'score_ref', 'created_at'
]
REJECT_COLS = [
    'row_id', 'symbol', 'candidate_row_id', 'method', 'subtype',
    'start_idx', 'end_idx', 'start_ts', 'end_ts', 'reason', 'created_at'
]

ACCEPTED = _load_or_init(ACCEPT_PATH_PARQUET, ACCEPT_COLS)
REJECTED = _load_or_init(REJECT_PATH_PARQUET, REJECT_COLS)

STATE = {
    'current_window': 0,
    'window_bars': int(WINDOW_BARS),
    'window_stride': int(WINDOW_STRIDE),
    'last_view_candidates': pd.DataFrame(),
    'last_action': None,
}

print(f'SYMBOL={SYMBOL} | rows={len(DF):,} | candidates={len(CAND):,} | accepted={len(ACCEPTED):,} | rejected={len(REJECTED):,}')



In [ ]:
# =========================
# Helpers de sesion
# =========================

def _now_iso() -> str:
    return datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ')


def _persist_all() -> None:
    ACCEPTED.to_parquet(ACCEPT_PATH_PARQUET, index=False)
    ACCEPTED.to_csv(ACCEPT_PATH_CSV, index=False)
    REJECTED.to_parquet(REJECT_PATH_PARQUET, index=False)
    REJECTED.to_csv(REJECT_PATH_CSV, index=False)


def _clip_idx(i: int) -> int:
    return int(min(max(i, 0), len(DF) - 1))


def _nearest_idx(ts_like) -> int:
    ts = pd.to_datetime(ts_like, errors='coerce')
    if pd.isna(ts):
        raise ValueError(f'timestamp invalido: {ts_like}')
    arr = DF['timestamp'].to_numpy()
    pos = int(np.searchsorted(arr, np.datetime64(ts), side='left'))
    if pos <= 0:
        return 0
    if pos >= len(arr):
        return len(arr) - 1
    before = arr[pos - 1]
    after = arr[pos]
    return pos if abs(after - np.datetime64(ts)) < abs(np.datetime64(ts) - before) else pos - 1


def _segment_row(start_idx: int, end_idx: int, label: str, quality: int = 2, notes: str = '',
                 source: str = 'manual_custom', candidate_row_id: str = None,
                 method: str = 'manual', subtype: str = 'manual', score_ref: float = np.nan) -> dict:
    s = _clip_idx(int(start_idx))
    e = _clip_idx(int(end_idx))
    if e < s:
        s, e = e, s

    return {
        'row_id': str(uuid.uuid4()),
        'symbol': SYMBOL,
        'source': source,
        'candidate_row_id': candidate_row_id,
        'method': method,
        'subtype': subtype,
        'start_idx': int(s),
        'end_idx': int(e),
        'start_ts': DF.loc[s, 'timestamp'],
        'end_ts': DF.loc[e, 'timestamp'],
        'n_bars': int(e - s + 1),
        'label': str(label),
        'quality': int(quality),
        'notes': str(notes),
        'score_ref': float(score_ref) if pd.notna(score_ref) else np.nan,
        'created_at': _now_iso(),
    }


def _exists_same_segment(start_idx: int, end_idx: int, label: str) -> bool:
    if ACCEPTED.empty:
        return False
    m = (
        (ACCEPTED['symbol'] == SYMBOL)
        & (ACCEPTED['start_idx'].astype(int) == int(start_idx))
        & (ACCEPTED['end_idx'].astype(int) == int(end_idx))
        & (ACCEPTED['label'].astype(str) == str(label))
    )
    return bool(m.any())


def session_status() -> None:
    print('--- Session Status ---')
    print(f'SYMBOL: {SYMBOL}')
    print(f'Rows data: {len(DF):,}')
    print(f'Candidatos: {len(CAND):,}')
    print(f'Aceptados: {len(ACCEPTED):,}')
    print(f'Rechazados: {len(REJECTED):,}')
    if len(ACCEPTED):
        print('\\nAceptados por label:')
        print(ACCEPTED['label'].value_counts().to_string())
    if STATE.get('last_action'):
        print(f"\nUltima accion: {STATE['last_action']}")


def list_labels(label: str = None, tail: int = 30):
    if ACCEPTED.empty:
        print('No hay labels aceptados aun.')
        return
    d = ACCEPTED.copy()
    if label is not None:
        d = d[d['label'].astype(str) == str(label)].copy()
    cols = ['created_at', 'label', 'quality', 'start_ts', 'end_ts', 'n_bars', 'method', 'subtype', 'source', 'notes', 'row_id']
    display(d[cols].tail(tail))

In [ ]:
# =========================
# Visualizacion y navegacion
# =========================

def _window_bounds(win_idx: int) -> tuple[int, int]:
    w = int(STATE['window_bars'])
    stride = int(STATE['window_stride'])
    s = int(win_idx * stride)
    s = _clip_idx(s)
    e = _clip_idx(s + w - 1)
    return s, e


def _window_candidates(win_s: int, win_e: int, top_n: int = 35) -> pd.DataFrame:
    if CAND.empty:
        return pd.DataFrame()
    c = CAND[(CAND['end_idx'] >= win_s) & (CAND['start_idx'] <= win_e)].copy()
    if c.empty:
        return c
    c['ov_start'] = c[['start_idx']].clip(lower=win_s)
    c['ov_end'] = c[['end_idx']].clip(upper=win_e)
    c['ov_bars'] = (c['ov_end'] - c['ov_start'] + 1).astype(int)
    c = c[c['ov_bars'] > 0].copy()
    c = c.sort_values('score', ascending=False).head(top_n).reset_index(drop=True)
    c.insert(0, 'local_id', np.arange(len(c), dtype=int))
    c['candidate_row_id'] = c.index.astype(str)
    return c


def view(win_idx: int = None, top_n: int = None, show_table: bool = True):
    if win_idx is not None:
        STATE['current_window'] = int(max(0, win_idx))
    if top_n is None:
        top_n = int(TOP_CANDIDATES_IN_VIEW)

    ws, we = _window_bounds(STATE['current_window'])
    sl = DF.iloc[ws:we+1].copy()

    cwin = _window_candidates(ws, we, top_n=top_n)
    STATE['last_view_candidates'] = cwin.copy()

    awin = ACCEPTED[(ACCEPTED['end_idx'].astype(int) >= ws) & (ACCEPTED['start_idx'].astype(int) <= we)].copy() if len(ACCEPTED) else pd.DataFrame()

    plt.figure(figsize=(PLOT_WIDTH, PLOT_HEIGHT))
    plt.plot(sl['timestamp'], sl['close'], color='black', lw=1.1)

    # candidatos
    color_map = {
        'triple_top': 'darkred',
        'triple_bottom': 'darkgreen',
        'trend_up': 'green',
        'trend_down': 'red',
        'impulse_up': 'limegreen',
        'impulse_down': 'tomato',
        'range_breakout_up': 'teal',
        'range_breakout_down': 'orange',
        'shock_up': 'navy',
        'shock_down': 'purple',
    }
    for _, r in cwin.iterrows():
        s = _clip_idx(int(r['start_idx']))
        e = _clip_idx(int(r['end_idx']))
        st = str(r.get('subtype', 'candidate'))
        col = color_map.get(st, 'gray')
        plt.axvspan(DF.loc[s, 'timestamp'], DF.loc[e, 'timestamp'], color=col, alpha=0.12)

    # aceptados manuales
    if len(awin):
        for _, r in awin.iterrows():
            s = _clip_idx(int(r['start_idx']))
            e = _clip_idx(int(r['end_idx']))
            plt.axvspan(DF.loc[s, 'timestamp'], DF.loc[e, 'timestamp'], color='gold', alpha=0.22)

    ts0 = DF.loc[ws, 'timestamp']
    ts1 = DF.loc[we, 'timestamp']
    plt.title(f'Manual Labeling - {SYMBOL} | win={STATE["current_window"]} | {ts0} -> {ts1} | cand={len(cwin)} | accepted_in_view={len(awin)}')
    plt.xlabel('timestamp')
    plt.ylabel('price')
    plt.grid(alpha=0.20)
    plt.tight_layout()
    plt.show()

    if show_table:
        if len(cwin):
            cols = ['local_id', 'subtype', 'method', 'score', 'start_ts', 'end_ts', 'n_bars', 'move_pips', 'range_pips', 'start_idx', 'end_idx']
            cols = [c for c in cols if c in cwin.columns]
            display(cwin[cols])
        else:
            print('No hay candidatos en esta ventana.')


def next_view(step: int = 1, **kwargs):
    STATE['current_window'] = max(0, int(STATE['current_window']) + int(step))
    return view(**kwargs)


def prev_view(step: int = 1, **kwargs):
    STATE['current_window'] = max(0, int(STATE['current_window']) - int(step))
    return view(**kwargs)


def jump_to_date(ts_like, **kwargs):
    i = _nearest_idx(ts_like)
    center = max(0, i - STATE['window_bars'] // 2)
    STATE['current_window'] = max(0, center // STATE['window_stride'])
    return view(**kwargs)


In [ ]:
# =========================
# Acciones de etiquetado manual
# =========================

def accept_candidate(local_id: int, label: str, quality: int = 2, notes: str = ''):
    global ACCEPTED
    c = STATE.get('last_view_candidates', pd.DataFrame())
    if c is None or c.empty:
        print('No hay candidatos cargados. Ejecuta view() primero.')
        return

    m = c[c['local_id'].astype(int) == int(local_id)]
    if m.empty:
        print(f'local_id={local_id} no existe en la vista actual.')
        return

    r = m.iloc[0]
    s = int(r['start_idx'])
    e = int(r['end_idx'])

    if _exists_same_segment(s, e, label):
        print('Segmento ya existe con ese label. No se agrega duplicado.')
        return

    row = _segment_row(
        start_idx=s,
        end_idx=e,
        label=label,
        quality=quality,
        notes=notes,
        source='candidate_accept',
        candidate_row_id=str(local_id),
        method=str(r.get('method', 'candidate')),
        subtype=str(r.get('subtype', 'candidate')),
        score_ref=float(r.get('score', np.nan)),
    )

    ACCEPTED = pd.concat([ACCEPTED, pd.DataFrame([row])], ignore_index=True)
    _persist_all()
    STATE['last_action'] = f"accept_candidate(local_id={local_id}, label='{label}')"
    print(f"OK aceptado: {label} | idx {s}->{e} | bars={row['n_bars']}")


def accept_custom(start_ts, end_ts, label: str, quality: int = 2, notes: str = ''):
    global ACCEPTED
    s = _nearest_idx(start_ts)
    e = _nearest_idx(end_ts)

    s2, e2 = (s, e) if s <= e else (e, s)
    if _exists_same_segment(s2, e2, label):
        print('Segmento custom ya existe con ese label. No se agrega duplicado.')
        return

    row = _segment_row(
        start_idx=s2,
        end_idx=e2,
        label=label,
        quality=quality,
        notes=notes,
        source='manual_custom',
        candidate_row_id=None,
        method='manual',
        subtype='manual',
        score_ref=np.nan,
    )

    ACCEPTED = pd.concat([ACCEPTED, pd.DataFrame([row])], ignore_index=True)
    _persist_all()
    STATE['last_action'] = f"accept_custom({start_ts}, {end_ts}, label='{label}')"
    print(f"OK custom: {label} | idx {row['start_idx']}->{row['end_idx']} | bars={row['n_bars']}")


def reject_candidate(local_id: int, reason: str = 'not_relevant'):
    global REJECTED
    c = STATE.get('last_view_candidates', pd.DataFrame())
    if c is None or c.empty:
        print('No hay candidatos cargados. Ejecuta view() primero.')
        return

    m = c[c['local_id'].astype(int) == int(local_id)]
    if m.empty:
        print(f'local_id={local_id} no existe en la vista actual.')
        return

    r = m.iloc[0]
    row = {
        'row_id': str(uuid.uuid4()),
        'symbol': SYMBOL,
        'candidate_row_id': str(local_id),
        'method': str(r.get('method', 'candidate')),
        'subtype': str(r.get('subtype', 'candidate')),
        'start_idx': int(r['start_idx']),
        'end_idx': int(r['end_idx']),
        'start_ts': pd.to_datetime(r.get('start_ts')),
        'end_ts': pd.to_datetime(r.get('end_ts')),
        'reason': str(reason),
        'created_at': _now_iso(),
    }

    REJECTED = pd.concat([REJECTED, pd.DataFrame([row])], ignore_index=True)
    _persist_all()
    STATE['last_action'] = f"reject_candidate(local_id={local_id}, reason='{reason}')"
    print(f"OK rechazado local_id={local_id} reason={reason}")


def undo_last_accept():
    global ACCEPTED
    if ACCEPTED.empty:
        print('No hay aceptados para deshacer.')
        return
    last = ACCEPTED.iloc[-1]
    ACCEPTED = ACCEPTED.iloc[:-1].reset_index(drop=True)
    _persist_all()
    STATE['last_action'] = 'undo_last_accept()'
    print(f"Undo OK: removido row_id={last['row_id']} label={last['label']}")


def export_train_ready(min_quality: int = 2):
    d = ACCEPTED.copy()
    if d.empty:
        print('No hay labels aceptados.')
        return
    d = d[d['quality'].astype(int) >= int(min_quality)].copy()
    out_pq = OUTPUT_DIR / f'{SYMBOL}_train_ready_q{min_quality}.parquet'
    out_csv = OUTPUT_DIR / f'{SYMBOL}_train_ready_q{min_quality}.csv'
    d.to_parquet(out_pq, index=False)
    d.to_csv(out_csv, index=False)
    print('Export train-ready:')
    print('-', out_pq)
    print('-', out_csv)
    print('rows:', len(d))


In [ ]:
# Inicio rapido
session_status()
view()


In [ ]:
# =========================
# UI interactiva (widgets)
# =========================

def _ts_fmt(i: int) -> str:
    return pd.Timestamp(DF.loc[_clip_idx(i), 'timestamp']).strftime('%Y-%m-%d %H:%M')


def launch_labeling_ui():
    if not HAS_WIDGETS:
        print('ipywidgets no esta disponible. Instala: pip install ipywidgets')
        return None

    label_options = [
        'trend_up', 'trend_down', 'triple_top', 'triple_bottom',
        'range', 'range_breakout_up', 'range_breakout_down',
        'reversal', 'continuation', 'impulse_up', 'impulse_down', 'other'
    ]

    max_win = max(0, int(np.ceil((len(DF) - 1) / max(1, int(STATE['window_stride'])))))

    w_win = widgets.IntSlider(value=int(STATE['current_window']), min=0, max=max_win, step=1, description='win', continuous_update=False)
    w_bars = widgets.IntSlider(value=int(STATE['window_bars']), min=80, max=720, step=20, description='bars', continuous_update=False)
    w_stride = widgets.IntSlider(value=int(STATE['window_stride']), min=20, max=360, step=10, description='stride', continuous_update=False)
    w_topn = widgets.IntSlider(value=int(TOP_CANDIDATES_IN_VIEW), min=0, max=120, step=5, description='top_n', continuous_update=False)

    b_prev = widgets.Button(description='Prev', button_style='')
    b_next = widgets.Button(description='Next', button_style='')
    b_prev10 = widgets.Button(description='-10', button_style='')
    b_next10 = widgets.Button(description='+10', button_style='')
    b_refresh = widgets.Button(description='Refresh', button_style='info')

    w_jump = widgets.Text(value='2016-06-01 00:00', description='jump_ts', placeholder='YYYY-MM-DD HH:MM')
    b_jump = widgets.Button(description='Jump', button_style='warning')

    w_local_id = widgets.BoundedIntText(value=0, min=0, max=9999, step=1, description='local_id')
    w_label = widgets.Combobox(options=label_options, value='trend_up', description='label', ensure_option=False)
    w_quality = widgets.Dropdown(options=[1, 2, 3, 4, 5], value=3, description='quality')
    w_notes = widgets.Text(value='', description='notes', placeholder='detalle opcional')
    b_accept = widgets.Button(description='Accept cand', button_style='success')
    b_reject = widgets.Button(description='Reject cand', button_style='danger')
    w_reject_reason = widgets.Text(value='not_relevant', description='reason')

    ws, we = _window_bounds(int(STATE['current_window']))
    w_start = widgets.Text(value=_ts_fmt(ws), description='start_ts', placeholder='YYYY-MM-DD HH:MM')
    w_end = widgets.Text(value=_ts_fmt(we), description='end_ts', placeholder='YYYY-MM-DD HH:MM')
    w_custom_label = widgets.Combobox(options=label_options, value='trend_up', description='custom_lbl', ensure_option=False)
    w_custom_quality = widgets.Dropdown(options=[1, 2, 3, 4, 5], value=3, description='quality')
    w_custom_notes = widgets.Text(value='', description='notes', placeholder='detalle opcional')
    b_use_left = widgets.Button(description='start=left', button_style='')
    b_use_right = widgets.Button(description='end=right', button_style='')
    b_accept_custom = widgets.Button(description='Accept custom', button_style='success')
    b_undo = widgets.Button(description='Undo last', button_style='')

    out_plot = widgets.Output()
    out_log = widgets.Output()

    def _sync_max_window():
        stride = max(1, int(w_stride.value))
        w_win.max = max(0, int(np.ceil((len(DF) - 1) / stride)))
        if w_win.value > w_win.max:
            w_win.value = w_win.max

    def _refresh(*_):
        STATE['window_bars'] = int(w_bars.value)
        STATE['window_stride'] = int(w_stride.value)
        STATE['current_window'] = int(w_win.value)
        _sync_max_window()

        with out_plot:
            clear_output(wait=True)
            view(win_idx=int(w_win.value), top_n=int(w_topn.value), show_table=True)

    def _log(msg: str):
        with out_log:
            print(msg)

    def _set_window_edge_texts():
        s, e = _window_bounds(int(w_win.value))
        w_start.value = _ts_fmt(s)
        w_end.value = _ts_fmt(e)

    def _on_prev(_):
        w_win.value = max(w_win.min, int(w_win.value) - 1)
        _set_window_edge_texts()
        _refresh()

    def _on_next(_):
        w_win.value = min(w_win.max, int(w_win.value) + 1)
        _set_window_edge_texts()
        _refresh()

    def _on_prev10(_):
        w_win.value = max(w_win.min, int(w_win.value) - 10)
        _set_window_edge_texts()
        _refresh()

    def _on_next10(_):
        w_win.value = min(w_win.max, int(w_win.value) + 10)
        _set_window_edge_texts()
        _refresh()

    def _on_jump(_):
        try:
            i = _nearest_idx(w_jump.value)
            center = max(0, i - int(w_bars.value) // 2)
            w_win.value = max(0, center // max(1, int(w_stride.value)))
            _set_window_edge_texts()
            _refresh()
        except Exception as ex:
            _log(f'Jump error: {ex}')

    def _on_accept(_):
        try:
            accept_candidate(local_id=int(w_local_id.value), label=str(w_label.value), quality=int(w_quality.value), notes=str(w_notes.value))
            _refresh()
        except Exception as ex:
            _log(f'Accept candidate error: {ex}')

    def _on_reject(_):
        try:
            reject_candidate(local_id=int(w_local_id.value), reason=str(w_reject_reason.value))
            _refresh()
        except Exception as ex:
            _log(f'Reject candidate error: {ex}')

    def _on_use_left(_):
        s, _ = _window_bounds(int(w_win.value))
        w_start.value = _ts_fmt(s)

    def _on_use_right(_):
        _, e = _window_bounds(int(w_win.value))
        w_end.value = _ts_fmt(e)

    def _on_accept_custom(_):
        try:
            accept_custom(
                start_ts=str(w_start.value),
                end_ts=str(w_end.value),
                label=str(w_custom_label.value),
                quality=int(w_custom_quality.value),
                notes=str(w_custom_notes.value),
            )
            _refresh()
        except Exception as ex:
            _log(f'Accept custom error: {ex}')

    def _on_undo(_):
        try:
            undo_last_accept()
            _refresh()
        except Exception as ex:
            _log(f'Undo error: {ex}')

    b_prev.on_click(_on_prev)
    b_next.on_click(_on_next)
    b_prev10.on_click(_on_prev10)
    b_next10.on_click(_on_next10)
    b_refresh.on_click(lambda _: _refresh())
    b_jump.on_click(_on_jump)
    b_accept.on_click(_on_accept)
    b_reject.on_click(_on_reject)
    b_use_left.on_click(_on_use_left)
    b_use_right.on_click(_on_use_right)
    b_accept_custom.on_click(_on_accept_custom)
    b_undo.on_click(_on_undo)

    w_win.observe(_refresh, names='value')
    w_bars.observe(lambda _: (_sync_max_window(), _refresh()), names='value')
    w_stride.observe(lambda _: (_sync_max_window(), _refresh()), names='value')
    w_topn.observe(_refresh, names='value')

    nav_row = widgets.HBox([b_prev10, b_prev, b_next, b_next10, b_refresh])
    jump_row = widgets.HBox([w_jump, b_jump])
    view_row = widgets.HBox([w_win, w_bars, w_stride, w_topn])

    cand_row1 = widgets.HBox([w_local_id, w_label, w_quality])
    cand_row2 = widgets.HBox([w_notes])
    cand_row3 = widgets.HBox([b_accept, w_reject_reason, b_reject, b_undo])

    custom_row1 = widgets.HBox([w_start, b_use_left, w_end, b_use_right])
    custom_row2 = widgets.HBox([w_custom_label, w_custom_quality])
    custom_row3 = widgets.HBox([w_custom_notes])
    custom_row4 = widgets.HBox([b_accept_custom])

    ui = widgets.VBox([
        widgets.HTML('<h3>Manual Labeling UI (2016)</h3>'),
        view_row,
        nav_row,
        jump_row,
        widgets.HTML('<b>Candidato en vista</b>'),
        cand_row1,
        cand_row2,
        cand_row3,
        widgets.HTML('<b>Segmento custom</b>'),
        custom_row1,
        custom_row2,
        custom_row3,
        custom_row4,
        widgets.HTML('<b>Grafico y candidatos de la ventana</b>'),
        out_plot,
        widgets.HTML('<b>Log</b>'),
        out_log,
    ])

    display(ui)
    _set_window_edge_texts()
    _refresh()

    return {
        'ui': ui,
        'controls': {
            'window': w_win,
            'bars': w_bars,
            'stride': w_stride,
            'top_n': w_topn,
            'jump': w_jump,
            'local_id': w_local_id,
            'label': w_label,
            'custom_start': w_start,
            'custom_end': w_end,
        },
        'outputs': {
            'plot': out_plot,
            'log': out_log,
        }
    }


UI = launch_labeling_ui()



## Ejemplos de uso (sin widgets)

```python
# Navegacion
next_view()
next_view(step=2)
prev_view()
jump_to_date('2016-12-15 10:00')

# Aceptar candidato de la tabla visible
accept_candidate(local_id=4, label='triple_top', quality=3, notes='muy claro')

# Rechazar candidato
reject_candidate(local_id=7, reason='ruido')

# Segmento custom manual por fechas
accept_custom('2016-11-10 04:00', '2016-11-12 22:00', label='trend_down', quality=3, notes='tendencia completa')

# Ver estado
session_status()
list_labels(tail=20)

# Export final para entrenamiento
export_train_ready(min_quality=2)
```

